# Notebook 03 - Calibration Model Development

**Milestone:** HV-006  
**Technical Gate:** CM-001  
**Primary Engineering Question:** Calibration Model Development

This notebook develops and freezes an RH calibration model using only qualified analyst-visible `STATIC / CALIBRATION` evidence from the frozen `VSM-LAB-DATA-v1.1` release.

The notebook must preserve the analyst-blind boundary established by DQ-001 and the calibration/validation separation required by repository governance.

**No VALIDATION measurement result is used for fitting, tuning, feature selection, or model selection in HV-006.**

## 1. Engineering Question

Can a calibration model built only from the frozen CALIBRATION subset reduce systematic RH measurement error while remaining:

- interpretable;
- free from experimental-unit leakage;
- operationally deployable; and
- suitable for later blind independent validation?

HV-006 answers the **model-development** question only.

The independent question—

> Does the frozen calibration generalize to reserved validation data?

—is deferred to Notebook 04.

## 2. Scientific Boundary

### Included

- `STATIC / CALIBRATION` measurement records only;
- public test-matrix design metadata;
- Test_ID-level calibration summaries;
- DUT-observable candidate predictors;
- calibration-only internal cross-validation;
- calibration-only residual diagnostics;
- final model freeze.

### Prohibited in HV-006

- VALIDATION measurement values or validation performance metrics;
- hidden simulator truth;
- private DUT personality/configuration;
- target-side/reference variables as deployable predictors;
- sample-level random train/test splitting;
- model choice informed by Notebook 04.

The public test matrix may be used to verify that a reserved validation design exists, but its measurement values remain outside the HV-006 evidence boundary.

## 3. Experimental Unit and Calibration Target

The primary model-development unit is **one summary record per CALIBRATION Test_ID**.

Each Test_ID contains 60 repeated 1 Hz measurements under one nominal static condition. Those 60 samples are useful for estimating the Test_ID mean and within-test scatter, but they are **not 60 independent calibration experiments**.

Therefore:

```text
5,760 CALIBRATION samples
        ↓ summarize by Test_ID
96 calibration modeling records
```

The pre-calibration error convention remains:

```text
Error_RH_Pre = RH_DUT_Mean - RH_Reference_Mean
```

The correction target is:

```text
Correction_RH = RH_Reference_Mean - RH_DUT_Mean
              = -Error_RH_Pre
```

Later, the calibrated result will be:

```text
RH_Calibrated = RH_DUT + Predicted_Correction_RH
```

and the post-calibration residual will use:

```text
Residual_Cal = RH_Calibrated - RH_Reference
```

The reference indication is a reference measurement, not exact physical truth.

## 4. Reproducible Setup

Reusable calibration loading and Test_ID-summary logic is kept in `src/calibration_analysis.py`.

This notebook remains the engineering record: it verifies the analysis boundary, shows the design, constructs the modeling table, and later documents model comparison and freeze decisions.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import sklearn

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


# ------------------------------------------------------------
# Locate project root and import reusable analyst-side helpers
# ------------------------------------------------------------

current_dir = Path.cwd()
candidate_roots = [current_dir, *current_dir.parents]

project_root = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw" / "vsm_lab_data_v1_1").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'data/raw/vsm_lab_data_v1_1'."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.calibration_analysis import (
    calibration_design_summary,
    load_static_calibration,
    summarize_calibration_test_ids,
)

reports_dir = project_root / "reports"
figures_dir = reports_dir / "figures"
tables_dir = reports_dir / "tables"

figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root : {project_root}")
print(f"Figures dir : {figures_dir}")
print(f"Tables dir  : {tables_dir}")

## 5. CM-01 — CALIBRATION Data Boundary Audit

The first executable requirement is to prove that the working measurement frame contains only `STATIC / CALIBRATION` evidence.

Expected calibration design from the frozen public test matrix:

```text
4 RH levels × 3 temperatures × 2 directions × 2 days × 2 runs
= 96 CALIBRATION Test_IDs

96 Test_IDs × 60 samples/Test_ID
= 5,760 measurement rows
```

The reserved validation design is checked only through the public test matrix. No validation measurement values are summarized or evaluated here.

In [ ]:
# ------------------------------------------------------------
# CM-01A: Load calibration evidence and public design metadata
# ------------------------------------------------------------

calibration_df, test_matrix_df = load_static_calibration(project_root)

calibration_matrix_df = test_matrix_df.loc[
    (test_matrix_df["Test_Family"] == "STATIC")
    & (test_matrix_df["Analysis_Role"] == "CALIBRATION")
].copy()

validation_matrix_df = test_matrix_df.loc[
    (test_matrix_df["Test_Family"] == "STATIC")
    & (test_matrix_df["Analysis_Role"] == "VALIDATION")
].copy()

samples_per_test = calibration_df.groupby("Test_ID").size()

print(f"CALIBRATION measurement rows : {len(calibration_df):,}")
print(f"CALIBRATION Test_IDs          : {calibration_df['Test_ID'].nunique()}")
print(f"CALIBRATION matrix Test_IDs   : {calibration_matrix_df['Test_ID'].nunique()}")
print(f"Reserved VALIDATION Test_IDs  : {validation_matrix_df['Test_ID'].nunique()}")
print(
    "Samples/Test_ID values       : "
    f"{sorted(samples_per_test.unique().tolist())}"
)
print(
    "Working Analysis_Role values : "
    f"{sorted(calibration_df['Analysis_Role'].unique().tolist())}"
)

assert len(calibration_df) == 5760
assert calibration_df["Test_ID"].nunique() == 96
assert calibration_matrix_df["Test_ID"].nunique() == 96
assert validation_matrix_df["Test_ID"].nunique() == 36
assert set(samples_per_test.unique()) == {60}
assert set(calibration_df["Test_Family"].unique()) == {"STATIC"}
assert set(calibration_df["Analysis_Role"].unique()) == {"CALIBRATION"}

calibration_measurement_ids = set(calibration_df["Test_ID"])
calibration_matrix_ids = set(calibration_matrix_df["Test_ID"])
validation_matrix_ids = set(validation_matrix_df["Test_ID"])

assert calibration_measurement_ids == calibration_matrix_ids
assert calibration_measurement_ids.isdisjoint(validation_matrix_ids)

print("\nCM-01 boundary audit: PASS")

## 6. CM-02 — Test_ID-Level Calibration Design

Now the 5,760 repeated samples are reduced to one traceable record per Test_ID.

The expected modeling design is:

- RH setpoints: 20, 40, 60, 80 %RH;
- temperature setpoints: 15, 25, 35 °C;
- Direction: Up / Down;
- Day: 1 / 2;
- Run: 1 / 2;
- 24 unique physical-condition groups defined by RH × Temperature × Direction;
- four Day × Run repetitions per physical-condition group.

This structure will later support group-aware cross-validation without allowing repetitions of the same physical condition to leak across training and assessment folds.

In [ ]:
# ------------------------------------------------------------
# CM-02A: Build one calibration record per Test_ID
# ------------------------------------------------------------

calibration_summary_df = summarize_calibration_test_ids(calibration_df)
design = calibration_design_summary(calibration_summary_df)

print("Calibration design audit")
print("------------------------")
for key, value in design.items():
    print(f"{key:28s}: {value}")

assert len(calibration_summary_df) == 96
assert design["N_Test_IDs"] == 96
assert design["RH_Levels"] == [20.0, 40.0, 60.0, 80.0]
assert design["Temperature_Levels_C"] == [15.0, 25.0, 35.0]
assert {str(x).upper() for x in design["Directions"]} == {"DOWN", "UP"}
assert design["Days"] == [1, 2]
assert design["Runs"] == [1, 2]
assert design["N_Condition_Groups"] == 24
assert design["Samples_Per_Test_ID_Unique"] == [60]

condition_replication = (
    calibration_summary_df
    .groupby("Condition_ID")
    .size()
)

assert set(condition_replication.unique()) == {4}

display(
    calibration_summary_df[
        [
            "Test_ID",
            "RH_Setpoint",
            "Temperature_Setpoint_C",
            "Direction",
            "Day_ID",
            "Run_ID",
            "N_Samples",
            "RH_DUT_Mean",
            "RH_Reference_Mean",
            "Temperature_DUT_Mean_C",
            "Error_RH_Pre",
            "Correction_RH",
            "Condition_ID",
        ]
    ].head(12).round(4)
)

print("\nCM-02 Test_ID summary audit: PASS")

## 7. CM-03 — Calibration Target Verification

Before any model is created, verify the sign convention algebraically and numerically.

For every Test_ID:

```text
Error_RH_Pre + Correction_RH = 0
```

because:

```text
Error_RH_Pre = RH_DUT_Mean - RH_Reference_Mean
Correction_RH = RH_Reference_Mean - RH_DUT_Mean
```

This check prevents a sign error from propagating through the entire calibration workflow.

In [ ]:
# ------------------------------------------------------------
# CM-03A: Verify correction-target sign convention
# ------------------------------------------------------------

correction_identity_error = (
    calibration_summary_df["Correction_RH"]
    - (
        calibration_summary_df["RH_Reference_Mean"]
        - calibration_summary_df["RH_DUT_Mean"]
    )
)

sign_identity_error = (
    calibration_summary_df["Error_RH_Pre"]
    + calibration_summary_df["Correction_RH"]
)

assert np.allclose(correction_identity_error, 0.0)
assert np.allclose(sign_identity_error, 0.0)

print(
    "Maximum correction-definition discrepancy : "
    f"{correction_identity_error.abs().max():.3e} %RH"
)
print(
    "Maximum Error + Correction discrepancy     : "
    f"{sign_identity_error.abs().max():.3e} %RH"
)
print("\nCM-03 correction-target verification: PASS")

## 8. CM-04 — Modeling Table and Predictor-Leakage Audit

The deployable model-selection track starts with only DUT-observable predictors:

```text
RH_DUT_Mean
Temperature_DUT_Mean_C
```

The target is:

```text
Correction_RH
```

The following variables remain available for design control or diagnostics but are **not deployable predictors**:

- `RH_Reference_Mean`;
- `Temperature_Reference_Mean_C`;
- `RH_Setpoint`;
- `Temperature_Setpoint_C`;
- `Day_ID`;
- `Run_ID`;
- `Test_ID`.

`Direction` is retained as a diagnostic design factor because HV-004 showed substantial UP/DOWN structure. It is not automatically eligible for the final deployable model.

In [ ]:
# ------------------------------------------------------------
# CM-04A: Define deployable predictors, target, groups, diagnostics
# ------------------------------------------------------------

deployable_predictors = [
    "RH_DUT_Mean",
    "Temperature_DUT_Mean_C",
]

target_column = "Correction_RH"
group_column = "Condition_ID"

diagnostic_columns = [
    "RH_Setpoint",
    "Temperature_Setpoint_C",
    "Direction",
    "Day_ID",
    "Run_ID",
    "Error_RH_Pre",
]

prohibited_predictor_columns = {
    "RH_Reference_Mean",
    "Temperature_Reference_Mean_C",
    "RH_Setpoint",
    "Temperature_Setpoint_C",
    "Day_ID",
    "Run_ID",
    "Test_ID",
}

assert set(deployable_predictors).isdisjoint(prohibited_predictor_columns)
assert target_column not in deployable_predictors
assert group_column not in deployable_predictors

X_deployable = calibration_summary_df[deployable_predictors].copy()
y_correction = calibration_summary_df[target_column].copy()
cv_groups = calibration_summary_df[group_column].copy()

modeling_table_df = calibration_summary_df[
    [
        "Test_ID",
        group_column,
        *deployable_predictors,
        target_column,
        *diagnostic_columns,
    ]
].copy()

assert len(X_deployable) == 96
assert len(y_correction) == 96
assert len(cv_groups) == 96
assert X_deployable.isna().sum().sum() == 0
assert y_correction.isna().sum() == 0
assert cv_groups.nunique() == 24

print("Deployable predictors :", deployable_predictors)
print("Target                :", target_column)
print("CV grouping variable  :", group_column)
print("Modeling rows         :", len(modeling_table_df))
print("Condition groups      :", cv_groups.nunique())

display(modeling_table_df.head(12).round(4))

print("\nCM-04 predictor-leakage audit: PASS")

## 9. Initial Curated Evidence Export

At this stage only the calibration-design summary is exported.

No model-comparison table, fitted coefficients, or validation result exists yet.

In [ ]:
# ------------------------------------------------------------
# Export initial HV-006 design summary
# ------------------------------------------------------------

design_summary_export = pd.DataFrame(
    [
        {"Item": "Measurement rows", "Value": len(calibration_df)},
        {"Item": "Calibration Test_IDs", "Value": design["N_Test_IDs"]},
        {"Item": "RH levels", "Value": ", ".join(map(str, design["RH_Levels"]))},
        {
            "Item": "Temperature levels C",
            "Value": ", ".join(map(str, design["Temperature_Levels_C"])),
        },
        {"Item": "Directions", "Value": ", ".join(design["Directions"])},
        {"Item": "Days", "Value": ", ".join(map(str, design["Days"]))},
        {"Item": "Runs", "Value": ", ".join(map(str, design["Runs"]))},
        {
            "Item": "Condition groups",
            "Value": design["N_Condition_Groups"],
        },
        {
            "Item": "Samples per Test_ID",
            "Value": ", ".join(
                map(str, design["Samples_Per_Test_ID_Unique"])
            ),
        },
        {
            "Item": "Reserved validation Test_IDs",
            "Value": validation_matrix_df["Test_ID"].nunique(),
        },
    ]
)

design_summary_path = (
    tables_dir / "hv006_calibration_design_summary.csv"
)

design_summary_export.to_csv(design_summary_path, index=False)

display(design_summary_export)
print(f"\nExported: {design_summary_path}")

## 10. Initial HV-006 Checkpoint

The CALIBRATION-only boundary, Test_ID summary, correction-target sign convention, and predictor-leakage checks have been implemented.

The required clean-kernel checkpoint is:

```text
CM-01 boundary audit: PASS
CM-02 Test_ID summary audit: PASS
CM-03 correction-target verification: PASS
CM-04 predictor-leakage audit: PASS
```

Once those four checks pass, the notebook may proceed to the baseline model stage below.

The next question is deliberately modest:

> How much improvement is obtained by doing nothing (C0) versus applying only one global offset (C1), before introducing RH- or temperature-dependent calibration structure?


## 11. Why C0 and C1 Come First

Before fitting slope, temperature, interaction, or polynomial terms, two baselines are required.

### C0 — Uncalibrated baseline

```text
Predicted_Correction_RH = 0
```

This reproduces the raw Test_ID-level DUT performance and establishes the error level that calibration must beat.

### C1 — Constant-offset calibration

```text
Predicted_Correction_RH = mean(training Correction_RH)
```

This asks whether one global additive correction is sufficient.

The C1 correction for each assessment fold must be estimated from **training rows only**. Using the overall mean correction before cross-validation would leak information from the assessment fold into the fitted offset.

HV-004 already indicated that a constant offset is unlikely to explain the full error structure, but C1 remains essential as a quantitative baseline.

## 12. CM-05 — Frozen Group-Aware Cross-Validation Structure

The primary internal comparison uses `GroupKFold` with:

```text
group = Condition_ID
Condition_ID = RH_Setpoint × Temperature_Setpoint_C × Direction
```

There are 24 physical-condition groups and four Day × Run Test_ID repetitions inside each group.

The fold rule is:

> all repetitions of one physical condition must remain together in either training or assessment data for a fold.

This prevents the model from being assessed on a condition whose near-replicates were used during fitting.

A six-fold split is used here:

```text
24 condition groups / 6 folds = 4 held-out condition groups per fold
4 Test_ID repetitions/group   = 16 held-out Test_IDs per fold
```

The same frozen split list will be reused for later C2-C4 comparisons so model rankings are made on identical assessment partitions.

In [ ]:
# ------------------------------------------------------------
# CM-05A: Freeze and audit grouped cross-validation partitions
# ------------------------------------------------------------

n_cv_splits = 6
group_cv = GroupKFold(n_splits=n_cv_splits)

cv_splits = list(
    group_cv.split(
        X_deployable,
        y_correction,
        groups=cv_groups,
    )
)

fold_audit_records = []
all_test_indices = []

for fold_id, (train_idx, test_idx) in enumerate(cv_splits, start=1):
    train_groups = set(cv_groups.iloc[train_idx])
    test_groups = set(cv_groups.iloc[test_idx])

    assert train_groups.isdisjoint(test_groups)

    all_test_indices.extend(test_idx.tolist())

    fold_audit_records.append(
        {
            "Fold": fold_id,
            "N_Train_Rows": len(train_idx),
            "N_Test_Rows": len(test_idx),
            "N_Train_Groups": len(train_groups),
            "N_Test_Groups": len(test_groups),
            "Test_RH_Levels": ", ".join(
                map(
                    lambda x: f"{x:g}",
                    sorted(
                        calibration_summary_df.iloc[test_idx][
                            "RH_Setpoint"
                        ].unique()
                    ),
                )
            ),
            "Test_Temperature_Levels_C": ", ".join(
                map(
                    lambda x: f"{x:g}",
                    sorted(
                        calibration_summary_df.iloc[test_idx][
                            "Temperature_Setpoint_C"
                        ].unique()
                    ),
                )
            ),
            "Test_Directions": ", ".join(
                sorted(
                    calibration_summary_df.iloc[test_idx][
                        "Direction"
                    ].astype(str).unique()
                )
            ),
        }
    )

fold_audit_df = pd.DataFrame(fold_audit_records)

assert len(cv_splits) == 6
assert sorted(all_test_indices) == list(range(len(calibration_summary_df)))
assert len(set(all_test_indices)) == len(calibration_summary_df)
assert set(fold_audit_df["N_Test_Rows"]) == {16}
assert set(fold_audit_df["N_Test_Groups"]) == {4}

display(fold_audit_df)

print("\nCM-05 grouped cross-validation audit: PASS")

## 13. Calibration Residual Metrics

For every candidate model, performance is evaluated from **out-of-fold calibrated predictions**.

The post-calibration residual is:

```text
Residual_Cal = RH_Calibrated - RH_Reference
```

The primary comparison metric is RMSE.

The notebook also reports:

- MBE — residual center / signed bias;
- MAE — average absolute residual magnitude;
- median residual;
- residual standard deviation;
- maximum absolute residual;
- R² for calibrated RH versus reference RH.

R² is descriptive only. Because this dataset spans a broad RH range, a high R² can coexist with practically important calibration error. It is therefore never used alone to select a model.

In [ ]:
# ------------------------------------------------------------
# Reusable metric helper for out-of-fold calibration results
# ------------------------------------------------------------

def calibration_metrics(
    residual: pd.Series | np.ndarray,
    rh_reference: pd.Series | np.ndarray,
    rh_calibrated: pd.Series | np.ndarray,
) -> dict[str, float]:
    residual_array = np.asarray(residual, dtype=float)
    reference_array = np.asarray(rh_reference, dtype=float)
    calibrated_array = np.asarray(rh_calibrated, dtype=float)

    return {
        "N": int(residual_array.size),
        "MBE": float(np.mean(residual_array)),
        "MAE": float(np.mean(np.abs(residual_array))),
        "RMSE": float(np.sqrt(np.mean(residual_array**2))),
        "Median_Error": float(np.median(residual_array)),
        "Residual_SD": float(np.std(residual_array, ddof=1)),
        "Max_Abs_Error": float(np.max(np.abs(residual_array))),
        "R2": float(r2_score(reference_array, calibrated_array)),
    }

## 14. C0 — Uncalibrated Baseline

C0 applies no correction:

```text
Predicted_Correction_C0 = 0
RH_Calibrated_C0        = RH_DUT_Mean
Residual_C0             = Error_RH_Pre
```

C0 is not a fitted calibration model. It is the baseline that all later candidates must improve upon.

In [ ]:
# ------------------------------------------------------------
# C0: Uncalibrated out-of-fold baseline
# ------------------------------------------------------------

baseline_results_df = calibration_summary_df[
    [
        "Test_ID",
        "Condition_ID",
        "RH_Setpoint",
        "Temperature_Setpoint_C",
        "Direction",
        "Day_ID",
        "Run_ID",
        "RH_DUT_Mean",
        "RH_Reference_Mean",
        "Error_RH_Pre",
        "Correction_RH",
    ]
].copy()

baseline_results_df["Predicted_Correction_C0"] = 0.0
baseline_results_df["RH_Calibrated_C0"] = baseline_results_df["RH_DUT_Mean"]
baseline_results_df["Residual_C0"] = (
    baseline_results_df["RH_Calibrated_C0"]
    - baseline_results_df["RH_Reference_Mean"]
)

assert np.allclose(
    baseline_results_df["Residual_C0"],
    baseline_results_df["Error_RH_Pre"],
)

c0_metrics = calibration_metrics(
    residual=baseline_results_df["Residual_C0"],
    rh_reference=baseline_results_df["RH_Reference_Mean"],
    rh_calibrated=baseline_results_df["RH_Calibrated_C0"],
)

display(pd.DataFrame([{"Model": "C0_Uncalibrated", **c0_metrics}]).round(4))

## 15. C1 — Cross-Validated Constant-Offset Calibration

For each fold, C1 estimates one correction from the training portion only:

```text
Offset_fold = mean(Correction_RH in training rows)
```

That frozen fold-specific offset is then applied to every Test_ID in the held-out fold.

This provides an honest out-of-fold estimate of what a single global offset can accomplish without allowing assessment information to influence the offset.

In [ ]:
# ------------------------------------------------------------
# C1: Out-of-fold constant-offset calibration
# ------------------------------------------------------------

c1_predicted_correction = np.full(
    len(calibration_summary_df),
    np.nan,
    dtype=float,
)

c1_fold_records = []

for fold_id, (train_idx, test_idx) in enumerate(cv_splits, start=1):
    training_correction = y_correction.iloc[train_idx]
    fold_offset = float(training_correction.mean())

    c1_predicted_correction[test_idx] = fold_offset

    c1_fold_records.append(
        {
            "Fold": fold_id,
            "Train_Mean_Correction_RH": fold_offset,
            "N_Train": len(train_idx),
            "N_Test": len(test_idx),
        }
    )

assert not np.isnan(c1_predicted_correction).any()

baseline_results_df["Predicted_Correction_C1"] = c1_predicted_correction
baseline_results_df["RH_Calibrated_C1"] = (
    baseline_results_df["RH_DUT_Mean"]
    + baseline_results_df["Predicted_Correction_C1"]
)
baseline_results_df["Residual_C1"] = (
    baseline_results_df["RH_Calibrated_C1"]
    - baseline_results_df["RH_Reference_Mean"]
)

c1_metrics = calibration_metrics(
    residual=baseline_results_df["Residual_C1"],
    rh_reference=baseline_results_df["RH_Reference_Mean"],
    rh_calibrated=baseline_results_df["RH_Calibrated_C1"],
)

c1_fold_df = pd.DataFrame(c1_fold_records)

display(c1_fold_df.round(4))
display(pd.DataFrame([{"Model": "C1_Constant_Offset", **c1_metrics}]).round(4))

## 16. C0 vs C1 — Baseline Comparison

The first calibration question is now answerable:

> Does one global additive correction materially improve the raw sensor result?

The comparison below uses the same 96 Test_ID records and reports C1 only from out-of-fold predictions.

In [ ]:
# ------------------------------------------------------------
# Compare C0 and C1 using out-of-fold results
# ------------------------------------------------------------

baseline_comparison_df = pd.DataFrame(
    [
        {"Model": "C0_Uncalibrated", **c0_metrics},
        {"Model": "C1_Constant_Offset", **c1_metrics},
    ]
)

baseline_comparison_df["RMSE_Improvement_vs_C0_pct"] = (
    100.0
    * (
        c0_metrics["RMSE"]
        - baseline_comparison_df["RMSE"]
    )
    / c0_metrics["RMSE"]
)

baseline_comparison_df["MAE_Improvement_vs_C0_pct"] = (
    100.0
    * (
        c0_metrics["MAE"]
        - baseline_comparison_df["MAE"]
    )
    / c0_metrics["MAE"]
)

display(
    baseline_comparison_df[
        [
            "Model",
            "N",
            "MBE",
            "MAE",
            "RMSE",
            "Residual_SD",
            "Max_Abs_Error",
            "R2",
            "RMSE_Improvement_vs_C0_pct",
            "MAE_Improvement_vs_C0_pct",
        ]
    ].round(4)
)

## 17. Baseline Residual Structure Check

A lower overall RMSE does not prove that C1 is an adequate calibration.

The residual means are therefore summarized by RH, temperature, and Direction.

If C1 removes only the global center while substantial condition-dependent residual structure remains, that is evidence that a more informative model such as C2 or C3 is justified.

In [ ]:
# ------------------------------------------------------------
# Condition-level residual structure for C0 and C1
# ------------------------------------------------------------

residual_by_rh_df = (
    baseline_results_df
    .groupby("RH_Setpoint", as_index=False)
    .agg(
        N=("Test_ID", "size"),
        C0_MBE=("Residual_C0", "mean"),
        C1_MBE=("Residual_C1", "mean"),
        C0_MAE=("Residual_C0", lambda x: x.abs().mean()),
        C1_MAE=("Residual_C1", lambda x: x.abs().mean()),
    )
)

residual_by_temperature_df = (
    baseline_results_df
    .groupby("Temperature_Setpoint_C", as_index=False)
    .agg(
        N=("Test_ID", "size"),
        C0_MBE=("Residual_C0", "mean"),
        C1_MBE=("Residual_C1", "mean"),
        C0_MAE=("Residual_C0", lambda x: x.abs().mean()),
        C1_MAE=("Residual_C1", lambda x: x.abs().mean()),
    )
)

residual_by_direction_df = (
    baseline_results_df
    .groupby("Direction", as_index=False)
    .agg(
        N=("Test_ID", "size"),
        C0_MBE=("Residual_C0", "mean"),
        C1_MBE=("Residual_C1", "mean"),
        C0_MAE=("Residual_C0", lambda x: x.abs().mean()),
        C1_MAE=("Residual_C1", lambda x: x.abs().mean()),
    )
)

print("Residual structure by RH setpoint")
display(residual_by_rh_df.round(4))

print("Residual structure by temperature")
display(residual_by_temperature_df.round(4))

print("Residual structure by Direction")
display(residual_by_direction_df.round(4))

## 18. Baseline-Model Checkpoint

Before C2 is introduced, verify that:

- `CM-05 grouped cross-validation audit: PASS` is reported;
- every Test_ID receives exactly one out-of-fold assessment prediction;
- C0 reproduces the original pre-calibration residual exactly;
- C1 estimates its offset from training rows only;
- C0 and C1 metrics are computed from Test_ID-level results;
- the residual summaries by RH, temperature, and Direction are visible.

### Interpretation rule

Do **not** choose C1 merely because its MBE is closer to zero.

A constant offset can center the residual distribution while leaving strong RH-, temperature-, or Direction-dependent structure.

The next stage, if these checks pass, is:

```text
C2 — RH-only linear correction
C3 — RH + DUT-temperature linear correction
```

Those models will reuse the exact same frozen `cv_splits` defined in CM-05.

## 19. Engineering Interpretation of C0 and C1

The out-of-fold baseline results establish two important facts.

### C1 materially improves the global error level

Relative to the uncalibrated C0 baseline:

- RMSE decreases from approximately **1.3045 %RH** to **0.5216 %RH**;
- MAE decreases from approximately **1.2267 %RH** to **0.4923 %RH**;
- both reductions are approximately **60%**;
- global MBE moves from approximately **-1.2267 %RH** to approximately **0 %RH**.

Therefore a large fraction of the raw static calibration error behaves like a global offset.

### But C1 is not an adequate final calibration

The global zero MBE hides strong conditional structure.

After C1:

- DOWN mean residual is approximately **+0.4923 %RH**;
- UP mean residual is approximately **-0.4923 %RH**.

Thus the UP-DOWN residual separation is approximately:

```text
-0.4923 - (+0.4923) = -0.9846 %RH
```

This is a large direction-dependent structure relative to the overall C1 RMSE.

The residual center by RH and temperature is much smaller than the Direction effect:

- RH-level MBE remains roughly within ±0.14 %RH;
- temperature-level MBE remains roughly within ±0.08 %RH.

Therefore C1 mainly corrects the overall center while leaving substantial hysteresis/direction structure unresolved.

### Why R² is not sufficient

C0 already has R² ≈ **0.9966** despite an MBE of approximately **-1.23 %RH**.

This demonstrates why R² is not a calibration-accuracy metric by itself. A sensor can track the reference almost perfectly over a wide RH range while still carrying a practically important systematic bias.

### C1 decision

**C1 is retained as a required baseline but is not accepted as the final calibration model.**

The next step is to test whether deployable continuous predictors explain additional error structure:

- C2 — RH-only linear correction;
- C3 — RH + DUT-temperature linear correction.

Direction remains diagnostic at this stage and is not yet introduced as a deployable predictor.

## 20. C2 — RH-Only Linear Correction

C2 fits a linear correction model using only the DUT RH indication:

```text
Correction_RH = beta0 + beta1 * RH_DUT_Mean
```

For each frozen group-CV fold:

1. fit the linear model on training Test_IDs only;
2. predict correction for held-out Test_IDs;
3. construct calibrated RH;
4. calculate out-of-fold residuals.

C2 tests whether the correction varies systematically with measured RH rather than being a single constant offset.

In [ ]:
# ------------------------------------------------------------
# C2: Out-of-fold RH-only linear correction
# ------------------------------------------------------------

c2_predicted_correction = np.full(
    len(calibration_summary_df),
    np.nan,
    dtype=float,
)

c2_fold_records = []

for fold_id, (train_idx, test_idx) in enumerate(cv_splits, start=1):
    X_train = calibration_summary_df.iloc[train_idx][["RH_DUT_Mean"]]
    y_train = y_correction.iloc[train_idx]
    X_test = calibration_summary_df.iloc[test_idx][["RH_DUT_Mean"]]

    model_c2 = LinearRegression()
    model_c2.fit(X_train, y_train)

    c2_predicted_correction[test_idx] = model_c2.predict(X_test)

    c2_fold_records.append(
        {
            "Fold": fold_id,
            "Intercept": float(model_c2.intercept_),
            "Beta_RH_DUT": float(model_c2.coef_[0]),
            "N_Train": len(train_idx),
            "N_Test": len(test_idx),
        }
    )

assert not np.isnan(c2_predicted_correction).any()

baseline_results_df["Predicted_Correction_C2"] = c2_predicted_correction
baseline_results_df["RH_Calibrated_C2"] = (
    baseline_results_df["RH_DUT_Mean"]
    + baseline_results_df["Predicted_Correction_C2"]
)
baseline_results_df["Residual_C2"] = (
    baseline_results_df["RH_Calibrated_C2"]
    - baseline_results_df["RH_Reference_Mean"]
)

c2_metrics = calibration_metrics(
    residual=baseline_results_df["Residual_C2"],
    rh_reference=baseline_results_df["RH_Reference_Mean"],
    rh_calibrated=baseline_results_df["RH_Calibrated_C2"],
)

c2_fold_df = pd.DataFrame(c2_fold_records)

display(c2_fold_df.round(6))
display(pd.DataFrame([{"Model": "C2_RH_Linear", **c2_metrics}]).round(4))

## 21. C3 — RH + DUT-Temperature Linear Correction

C3 adds the DUT temperature indication as a second deployable predictor:

```text
Correction_RH
=
beta0
+ beta1 * RH_DUT_Mean
+ beta2 * Temperature_DUT_Mean_C
```

This model tests whether temperature contributes calibration information beyond the RH-only C2 model.

The same frozen group-CV folds are reused so C1, C2, and C3 are compared on identical held-out conditions.

In [ ]:
# ------------------------------------------------------------
# C3: Out-of-fold RH + DUT-temperature linear correction
# ------------------------------------------------------------

c3_predictors = [
    "RH_DUT_Mean",
    "Temperature_DUT_Mean_C",
]

c3_predicted_correction = np.full(
    len(calibration_summary_df),
    np.nan,
    dtype=float,
)

c3_fold_records = []

for fold_id, (train_idx, test_idx) in enumerate(cv_splits, start=1):
    X_train = calibration_summary_df.iloc[train_idx][c3_predictors]
    y_train = y_correction.iloc[train_idx]
    X_test = calibration_summary_df.iloc[test_idx][c3_predictors]

    model_c3 = LinearRegression()
    model_c3.fit(X_train, y_train)

    c3_predicted_correction[test_idx] = model_c3.predict(X_test)

    c3_fold_records.append(
        {
            "Fold": fold_id,
            "Intercept": float(model_c3.intercept_),
            "Beta_RH_DUT": float(model_c3.coef_[0]),
            "Beta_Temperature_DUT_C": float(model_c3.coef_[1]),
            "N_Train": len(train_idx),
            "N_Test": len(test_idx),
        }
    )

assert not np.isnan(c3_predicted_correction).any()

baseline_results_df["Predicted_Correction_C3"] = c3_predicted_correction
baseline_results_df["RH_Calibrated_C3"] = (
    baseline_results_df["RH_DUT_Mean"]
    + baseline_results_df["Predicted_Correction_C3"]
)
baseline_results_df["Residual_C3"] = (
    baseline_results_df["RH_Calibrated_C3"]
    - baseline_results_df["RH_Reference_Mean"]
)

c3_metrics = calibration_metrics(
    residual=baseline_results_df["Residual_C3"],
    rh_reference=baseline_results_df["RH_Reference_Mean"],
    rh_calibrated=baseline_results_df["RH_Calibrated_C3"],
)

c3_fold_df = pd.DataFrame(c3_fold_records)

display(c3_fold_df.round(6))
display(pd.DataFrame([{"Model": "C3_RH_Temperature_Linear", **c3_metrics}]).round(4))

## 22. C0-C3 Out-of-Fold Model Comparison

C0 through C3 are now compared using the same Test_ID-level evidence.

Interpretation must prioritize:

1. RMSE;
2. MAE;
3. absolute MBE;
4. residual SD and maximum absolute error;
5. residual structure;
6. parsimony and deployability.

A smaller numerical improvement is not automatically sufficient to justify an additional predictor.

In [ ]:
# ------------------------------------------------------------
# Compare C0-C3 on identical out-of-fold Test_ID results
# ------------------------------------------------------------

c0_c3_comparison_df = pd.DataFrame(
    [
        {"Model": "C0_Uncalibrated", **c0_metrics},
        {"Model": "C1_Constant_Offset", **c1_metrics},
        {"Model": "C2_RH_Linear", **c2_metrics},
        {"Model": "C3_RH_Temperature_Linear", **c3_metrics},
    ]
)

c0_c3_comparison_df["RMSE_Improvement_vs_C0_pct"] = (
    100.0
    * (
        c0_metrics["RMSE"]
        - c0_c3_comparison_df["RMSE"]
    )
    / c0_metrics["RMSE"]
)

c0_c3_comparison_df["RMSE_Improvement_vs_C1_pct"] = (
    100.0
    * (
        c1_metrics["RMSE"]
        - c0_c3_comparison_df["RMSE"]
    )
    / c1_metrics["RMSE"]
)

display(
    c0_c3_comparison_df[
        [
            "Model",
            "N",
            "MBE",
            "MAE",
            "RMSE",
            "Residual_SD",
            "Max_Abs_Error",
            "R2",
            "RMSE_Improvement_vs_C0_pct",
            "RMSE_Improvement_vs_C1_pct",
        ]
    ].round(4)
)

## 23. C2/C3 Residual Structure Diagnostics

Overall metrics alone cannot determine whether C2 or C3 has removed meaningful structure.

The following summaries compare C1, C2, and C3 residual center by:

- RH setpoint;
- temperature setpoint;
- Direction.

The key question is whether RH and temperature predictors reduce structure without simply redistributing error across conditions.

In [ ]:
# ------------------------------------------------------------
# Residual structure after C1, C2, and C3
# ------------------------------------------------------------

def residual_group_summary(
    df: pd.DataFrame,
    group_column: str,
) -> pd.DataFrame:
    return (
        df.groupby(group_column, as_index=False)
        .agg(
            N=("Test_ID", "size"),
            C1_MBE=("Residual_C1", "mean"),
            C2_MBE=("Residual_C2", "mean"),
            C3_MBE=("Residual_C3", "mean"),
            C1_MAE=("Residual_C1", lambda x: x.abs().mean()),
            C2_MAE=("Residual_C2", lambda x: x.abs().mean()),
            C3_MAE=("Residual_C3", lambda x: x.abs().mean()),
        )
    )

c123_by_rh_df = residual_group_summary(
    baseline_results_df,
    "RH_Setpoint",
)

c123_by_temperature_df = residual_group_summary(
    baseline_results_df,
    "Temperature_Setpoint_C",
)

c123_by_direction_df = residual_group_summary(
    baseline_results_df,
    "Direction",
)

print("C1-C3 residual structure by RH")
display(c123_by_rh_df.round(4))

print("C1-C3 residual structure by temperature")
display(c123_by_temperature_df.round(4))

print("C1-C3 residual structure by Direction")
display(c123_by_direction_df.round(4))

## 24. C2/C3 Checkpoint

At this checkpoint, do not select the final calibration model yet.

Review:

- C0-C3 out-of-fold RMSE and MAE;
- coefficient stability across folds;
- incremental benefit of temperature in C3 relative to C2;
- residual structure versus RH and temperature;
- remaining Direction-dependent residual.

If C2/C3 materially improve deployable calibration while Direction structure remains, the next stage will compare:

```text
C4 — low-order nonlinear deployable correction
C5 — Direction-augmented diagnostic model
```

C5 remains diagnostic unless Direction can be defined operationally for field use.

## 25. Engineering Interpretation of C2 and C3

The out-of-fold results show that neither deployable linear extension improves on C1.

### C2 — RH-only linear correction

Relative to C1:

- RMSE changes from approximately **0.5216** to **0.5243 %RH**;
- MAE changes only from approximately **0.4923** to **0.4914 %RH**;
- RMSE is therefore slightly worse, while MAE improves only trivially;
- the fitted RH slope is very small and changes in magnitude across folds.

C2 therefore does not provide a meaningful improvement over the simpler constant-offset model.

### C3 — RH + DUT-temperature linear correction

C3 performs materially worse:

- RMSE increases to approximately **0.6622 %RH**;
- MAE increases to approximately **0.6316 %RH**;
- the temperature coefficient changes sign across folds;
- Direction-dependent residual structure becomes even larger.

The sign instability of the temperature coefficient indicates that the apparent temperature contribution is not robust under the frozen group-aware assessment partitions.

### Direction remains the dominant unresolved structure

After C2:

- DOWN mean residual ≈ **+0.4915 %RH**;
- UP mean residual ≈ **-0.4914 %RH**.

After C3:

- DOWN mean residual ≈ **+0.6331 %RH**;
- UP mean residual ≈ **-0.6301 %RH**.

Therefore the principal unresolved calibration structure is not explained by simple RH or temperature dependence.

### C2/C3 decision

- **C2 is rejected as a final model because it adds complexity without improving RMSE.**
- **C3 is rejected because it materially worsens out-of-fold performance and shows unstable temperature coefficients.**
- **C1 remains the best deployable model among C0-C3.**

The next step is intentionally diagnostic:

1. test whether a low-order nonlinear deployable model C4 can outperform C1;
2. quantify how much performance changes when Direction is explicitly supplied in C5.

C5 is not automatically deployable. It is used first to determine whether the remaining error is fundamentally hysteresis/state-related.

## 26. C4 — Low-Order Nonlinear Deployable Correction

C4 keeps the same deployable inputs:

```text
RH_DUT_Mean
Temperature_DUT_Mean_C
```

but introduces second-order structure:

```text
RH_DUT
Temperature_DUT
RH_DUT^2
RH_DUT × Temperature_DUT
Temperature_DUT^2
```

A Ridge model is used to control coefficient instability.

### Nested alpha selection

The Ridge regularization parameter is selected **inside each outer training fold only** using an inner group-aware cross-validation.

This is required because choosing `alpha` from all 96 Test_IDs before outer assessment would leak information from held-out conditions into model tuning.

The alpha grid is deliberately small:

```text
0.01, 0.1, 1, 10, 100
```

C4 remains a deployable candidate because it uses only DUT-observable RH and temperature.

In [ ]:
# ------------------------------------------------------------
# C4: Degree-2 deployable polynomial correction with nested
#     group-aware Ridge alpha selection
# ------------------------------------------------------------

c4_predictors = [
    "RH_DUT_Mean",
    "Temperature_DUT_Mean_C",
]

c4_alpha_grid = [0.01, 0.1, 1.0, 10.0, 100.0]

c4_predicted_correction = np.full(
    len(calibration_summary_df),
    np.nan,
    dtype=float,
)

c4_fold_records = []

for outer_fold_id, (train_idx, test_idx) in enumerate(cv_splits, start=1):
    outer_train_df = calibration_summary_df.iloc[train_idx].copy()
    outer_test_df = calibration_summary_df.iloc[test_idx].copy()

    X_outer_train = outer_train_df[c4_predictors]
    y_outer_train = outer_train_df[target_column]
    groups_outer_train = outer_train_df[group_column]

    X_outer_test = outer_test_df[c4_predictors]

    inner_cv = GroupKFold(n_splits=4)

    alpha_scores = []

    for alpha in c4_alpha_grid:
        inner_squared_errors = []

        for inner_train_idx, inner_valid_idx in inner_cv.split(
            X_outer_train,
            y_outer_train,
            groups=groups_outer_train,
        ):
            model_c4_inner = Pipeline(
                steps=[
                    (
                        "poly",
                        PolynomialFeatures(
                            degree=2,
                            include_bias=False,
                        ),
                    ),
                    ("scale", StandardScaler()),
                    ("ridge", Ridge(alpha=alpha)),
                ]
            )

            model_c4_inner.fit(
                X_outer_train.iloc[inner_train_idx],
                y_outer_train.iloc[inner_train_idx],
            )

            inner_pred = model_c4_inner.predict(
                X_outer_train.iloc[inner_valid_idx]
            )

            inner_residual = (
                inner_pred
                - y_outer_train.iloc[inner_valid_idx].to_numpy()
            )

            inner_squared_errors.extend(
                (inner_residual**2).tolist()
            )

        alpha_scores.append(
            {
                "Alpha": alpha,
                "Inner_RMSE": float(
                    np.sqrt(np.mean(inner_squared_errors))
                ),
            }
        )

    alpha_score_df = pd.DataFrame(alpha_scores)
    best_alpha = float(
        alpha_score_df
        .sort_values(["Inner_RMSE", "Alpha"])
        .iloc[0]["Alpha"]
    )

    model_c4 = Pipeline(
        steps=[
            (
                "poly",
                PolynomialFeatures(
                    degree=2,
                    include_bias=False,
                ),
            ),
            ("scale", StandardScaler()),
            ("ridge", Ridge(alpha=best_alpha)),
        ]
    )

    model_c4.fit(X_outer_train, y_outer_train)

    c4_predicted_correction[test_idx] = model_c4.predict(X_outer_test)

    c4_fold_records.append(
        {
            "Fold": outer_fold_id,
            "Selected_Alpha": best_alpha,
            "Best_Inner_RMSE": float(
                alpha_score_df["Inner_RMSE"].min()
            ),
            "N_Train": len(train_idx),
            "N_Test": len(test_idx),
        }
    )

assert not np.isnan(c4_predicted_correction).any()

baseline_results_df["Predicted_Correction_C4"] = c4_predicted_correction
baseline_results_df["RH_Calibrated_C4"] = (
    baseline_results_df["RH_DUT_Mean"]
    + baseline_results_df["Predicted_Correction_C4"]
)
baseline_results_df["Residual_C4"] = (
    baseline_results_df["RH_Calibrated_C4"]
    - baseline_results_df["RH_Reference_Mean"]
)

c4_metrics = calibration_metrics(
    residual=baseline_results_df["Residual_C4"],
    rh_reference=baseline_results_df["RH_Reference_Mean"],
    rh_calibrated=baseline_results_df["RH_Calibrated_C4"],
)

c4_fold_df = pd.DataFrame(c4_fold_records)

display(c4_fold_df.round(6))
display(pd.DataFrame([{"Model": "C4_Polynomial_Ridge", **c4_metrics}]).round(4))

## 27. C5 — Direction-Augmented Diagnostic Model

C5 is designed to answer a different question:

> How much of the remaining calibration error can be removed if Direction is explicitly known?

C5 uses a simple binary Direction indicator in addition to the constant model:

```text
Correction_RH
=
beta0
+ beta1 × I(Direction = UP)
```

where:

```text
I(Direction = UP) = 1 for UP
                  = 0 for DOWN
```

This deliberately isolates the value of Direction information.

### Important limitation

C5 is **diagnostic by default**.

Direction is an experimental-state variable. It is not accepted as a deployable predictor unless a later engineering decision defines a reliable field rule for determining whether the sensor is on an increasing or decreasing RH trajectory.

In [ ]:
# ------------------------------------------------------------
# C5: Direction-augmented diagnostic model
# ------------------------------------------------------------

direction_up = (
    calibration_summary_df["Direction"]
    .astype(str)
    .str.upper()
    .eq("UP")
    .astype(int)
)

X_c5 = pd.DataFrame(
    {
        "Direction_UP": direction_up,
    },
    index=calibration_summary_df.index,
)

c5_predicted_correction = np.full(
    len(calibration_summary_df),
    np.nan,
    dtype=float,
)

c5_fold_records = []

for fold_id, (train_idx, test_idx) in enumerate(cv_splits, start=1):
    model_c5 = LinearRegression()

    model_c5.fit(
        X_c5.iloc[train_idx],
        y_correction.iloc[train_idx],
    )

    c5_predicted_correction[test_idx] = model_c5.predict(
        X_c5.iloc[test_idx]
    )

    c5_fold_records.append(
        {
            "Fold": fold_id,
            "Intercept_DOWN_Correction": float(model_c5.intercept_),
            "UP_Increment": float(model_c5.coef_[0]),
            "N_Train": len(train_idx),
            "N_Test": len(test_idx),
        }
    )

assert not np.isnan(c5_predicted_correction).any()

baseline_results_df["Predicted_Correction_C5"] = c5_predicted_correction
baseline_results_df["RH_Calibrated_C5"] = (
    baseline_results_df["RH_DUT_Mean"]
    + baseline_results_df["Predicted_Correction_C5"]
)
baseline_results_df["Residual_C5"] = (
    baseline_results_df["RH_Calibrated_C5"]
    - baseline_results_df["RH_Reference_Mean"]
)

c5_metrics = calibration_metrics(
    residual=baseline_results_df["Residual_C5"],
    rh_reference=baseline_results_df["RH_Reference_Mean"],
    rh_calibrated=baseline_results_df["RH_Calibrated_C5"],
)

c5_fold_df = pd.DataFrame(c5_fold_records)

display(c5_fold_df.round(6))
display(pd.DataFrame([{"Model": "C5_Direction_Diagnostic", **c5_metrics}]).round(4))

## 28. C0-C5 Out-of-Fold Comparison

This table separates two questions:

### Deployable model competition

```text
C1
C2
C3
C4
```

These use no explicit Direction state.

### Diagnostic ceiling

```text
C5
```

C5 quantifies what becomes possible if Direction information is explicitly available.

A strong C5 improvement does **not** by itself authorize a direction-aware final calibration. It instead identifies hysteresis/state dependence as an engineering limitation that must be addressed explicitly.

In [ ]:
# ------------------------------------------------------------
# Compare C0-C5 using identical outer group-CV assessment
# ------------------------------------------------------------

c0_c5_comparison_df = pd.DataFrame(
    [
        {"Model": "C0_Uncalibrated", "Role": "Baseline", **c0_metrics},
        {"Model": "C1_Constant_Offset", "Role": "Deployable", **c1_metrics},
        {"Model": "C2_RH_Linear", "Role": "Deployable", **c2_metrics},
        {
            "Model": "C3_RH_Temperature_Linear",
            "Role": "Deployable",
            **c3_metrics,
        },
        {
            "Model": "C4_Polynomial_Ridge",
            "Role": "Deployable",
            **c4_metrics,
        },
        {
            "Model": "C5_Direction_Diagnostic",
            "Role": "Diagnostic",
            **c5_metrics,
        },
    ]
)

c0_c5_comparison_df["RMSE_Improvement_vs_C0_pct"] = (
    100.0
    * (
        c0_metrics["RMSE"]
        - c0_c5_comparison_df["RMSE"]
    )
    / c0_metrics["RMSE"]
)

c0_c5_comparison_df["RMSE_Improvement_vs_C1_pct"] = (
    100.0
    * (
        c1_metrics["RMSE"]
        - c0_c5_comparison_df["RMSE"]
    )
    / c1_metrics["RMSE"]
)

display(
    c0_c5_comparison_df[
        [
            "Model",
            "Role",
            "N",
            "MBE",
            "MAE",
            "RMSE",
            "Residual_SD",
            "Max_Abs_Error",
            "R2",
            "RMSE_Improvement_vs_C0_pct",
            "RMSE_Improvement_vs_C1_pct",
        ]
    ].round(4)
)

## 29. Direction Diagnostic After C4 and C5

The final diagnostic at this stage compares mean residual by Direction.

Interpretation:

- if C4 still shows approximately equal-and-opposite UP/DOWN residual means, nonlinear RH/temperature structure has not solved the dominant problem;
- if C5 strongly collapses the UP/DOWN split, the evidence identifies Direction/hysteresis as the principal remaining systematic structure.

In [ ]:
# ------------------------------------------------------------
# Direction residual structure after C1, C4, and C5
# ------------------------------------------------------------

c145_by_direction_df = (
    baseline_results_df
    .groupby("Direction", as_index=False)
    .agg(
        N=("Test_ID", "size"),
        C1_MBE=("Residual_C1", "mean"),
        C4_MBE=("Residual_C4", "mean"),
        C5_MBE=("Residual_C5", "mean"),
        C1_MAE=("Residual_C1", lambda x: x.abs().mean()),
        C4_MAE=("Residual_C4", lambda x: x.abs().mean()),
        C5_MAE=("Residual_C5", lambda x: x.abs().mean()),
    )
)

display(c145_by_direction_df.round(4))

## 30. C4/C5 Checkpoint

Do not freeze the final model immediately after this section.

Review:

- C4 out-of-fold RMSE relative to C1;
- C4 selected Ridge alpha across outer folds;
- whether C4 materially reduces residual structure;
- C5 out-of-fold RMSE and MAE;
- the C5 Direction coefficient;
- the remaining C5 residual by Direction.

### Decision logic

If C4 does not materially outperform C1, the simpler C1 remains the leading deployable model.

If C5 dramatically outperforms all deployable models, the correct engineering conclusion is not simply “choose C5.” Instead:

> explicit Direction/state information contains major calibration value, so hysteresis must either be operationalized as a stateful field input or documented as a limitation of a memoryless calibration.

After this checkpoint, the selected deployable candidate and C5 diagnostic model will undergo the planned leave-one-RH and leave-one-day robustness stress tests before model freeze.

## 31. Engineering Interpretation of C4 and C5

The C4/C5 checkpoint resolves the main model-development question.

### C4 — nonlinear deployable model

C4 does not improve on C1:

- C1 RMSE ≈ **0.5216 %RH**;
- C4 RMSE ≈ **0.6509 %RH**;
- C4 is approximately **24.8% worse than C1** in RMSE;
- C4 MAE ≈ **0.6249 %RH**;
- C4 retains a strong equal-and-opposite Direction residual:
  - DOWN ≈ **+0.6329 %RH**;
  - UP ≈ **-0.6169 %RH**.

Therefore second-order RH/temperature structure does not explain the dominant remaining calibration error.

### C5 — Direction diagnostic

C5 produces a major out-of-fold improvement:

- RMSE ≈ **0.1779 %RH**;
- MAE ≈ **0.1449 %RH**;
- maximum absolute residual ≈ **0.4152 %RH**;
- RMSE improves by approximately **65.9% relative to C1**;
- RMSE improves by approximately **86.4% relative to C0**.

Most importantly, the Direction-conditioned residual means collapse to approximately zero for both UP and DOWN.

This is strong evidence that the dominant unresolved systematic structure is **Direction-linked and consistent with hysteresis/state dependence**.

### Model-development decision at this checkpoint

Among memoryless deployable models C1-C4, **C1 remains the preferred candidate** because it is simpler and performs better than the added RH/temperature models.

C5 is retained as a **diagnostic ceiling / state-aware candidate**, not automatically as the deployable calibration model.

The remaining question is robustness:

> Does the C1 conclusion, and the large C5 advantage, remain when entire RH levels or entire days are withheld from fitting?

## 32. Robustness Stress Tests

Two pre-specified stress tests are now applied to C1 and C5.

### Stress Test A — Leave-One-RH-Setpoint-Out

Each calibration RH setpoint is held out in turn:

```text
20, 40, 60, 80 %RH
```

The model is fit on the other three RH levels and assessed only on the held-out level.

This probes robustness to unobserved RH conditions within the calibration design. The edge levels 20 and 80 %RH are especially stringent because their assessment is effectively outside the remaining training RH range.

### Stress Test B — Leave-One-Day-Out

Calibration contains Day 1 and Day 2.

Each day is held out in turn:

```text
Train Day 1 → assess Day 2
Train Day 2 → assess Day 1
```

This probes robustness to the day-to-day intermediate-precision shift already identified in HV-005.

Only CALIBRATION data are used. Reserved VALIDATION measurements remain untouched.

In [ ]:
# ------------------------------------------------------------
# Stress Test A: Leave-one-RH-setpoint-out for C1 and C5
# ------------------------------------------------------------

rh_levels = sorted(calibration_summary_df["RH_Setpoint"].unique())

loo_rh_predictions_c1 = np.full(len(calibration_summary_df), np.nan, dtype=float)
loo_rh_predictions_c5 = np.full(len(calibration_summary_df), np.nan, dtype=float)

loo_rh_fold_records = []

for held_out_rh in rh_levels:
    test_mask = calibration_summary_df["RH_Setpoint"].eq(held_out_rh)
    train_mask = ~test_mask

    train_idx = np.flatnonzero(train_mask.to_numpy())
    test_idx = np.flatnonzero(test_mask.to_numpy())

    # C1: training-only global offset
    c1_offset = float(y_correction.iloc[train_idx].mean())
    loo_rh_predictions_c1[test_idx] = c1_offset

    # C5: training-only Direction diagnostic
    model_c5_rh = LinearRegression()
    model_c5_rh.fit(
        X_c5.iloc[train_idx],
        y_correction.iloc[train_idx],
    )
    loo_rh_predictions_c5[test_idx] = model_c5_rh.predict(
        X_c5.iloc[test_idx]
    )

    residual_c1_fold = (
        calibration_summary_df.iloc[test_idx]["Error_RH_Pre"].to_numpy()
        + loo_rh_predictions_c1[test_idx]
    )
    residual_c5_fold = (
        calibration_summary_df.iloc[test_idx]["Error_RH_Pre"].to_numpy()
        + loo_rh_predictions_c5[test_idx]
    )

    loo_rh_fold_records.append(
        {
            "Held_Out_RH": float(held_out_rh),
            "N_Test": len(test_idx),
            "C1_Offset": c1_offset,
            "C1_RMSE": float(np.sqrt(np.mean(residual_c1_fold**2))),
            "C1_MAE": float(np.mean(np.abs(residual_c1_fold))),
            "C5_DOWN_Correction": float(model_c5_rh.intercept_),
            "C5_UP_Increment": float(model_c5_rh.coef_[0]),
            "C5_RMSE": float(np.sqrt(np.mean(residual_c5_fold**2))),
            "C5_MAE": float(np.mean(np.abs(residual_c5_fold))),
        }
    )

assert not np.isnan(loo_rh_predictions_c1).any()
assert not np.isnan(loo_rh_predictions_c5).any()

loo_rh_results_df = calibration_summary_df[
    [
        "Test_ID",
        "RH_Setpoint",
        "Temperature_Setpoint_C",
        "Direction",
        "Day_ID",
        "Run_ID",
        "RH_DUT_Mean",
        "RH_Reference_Mean",
        "Error_RH_Pre",
    ]
].copy()

loo_rh_results_df["Predicted_Correction_C1"] = loo_rh_predictions_c1
loo_rh_results_df["Predicted_Correction_C5"] = loo_rh_predictions_c5

loo_rh_results_df["Residual_C1"] = (
    loo_rh_results_df["Error_RH_Pre"]
    + loo_rh_results_df["Predicted_Correction_C1"]
)
loo_rh_results_df["Residual_C5"] = (
    loo_rh_results_df["Error_RH_Pre"]
    + loo_rh_results_df["Predicted_Correction_C5"]
)

loo_rh_c1_metrics = calibration_metrics(
    residual=loo_rh_results_df["Residual_C1"],
    rh_reference=loo_rh_results_df["RH_Reference_Mean"],
    rh_calibrated=(
        loo_rh_results_df["RH_DUT_Mean"]
        + loo_rh_results_df["Predicted_Correction_C1"]
    ),
)

loo_rh_c5_metrics = calibration_metrics(
    residual=loo_rh_results_df["Residual_C5"],
    rh_reference=loo_rh_results_df["RH_Reference_Mean"],
    rh_calibrated=(
        loo_rh_results_df["RH_DUT_Mean"]
        + loo_rh_results_df["Predicted_Correction_C5"]
    ),
)

loo_rh_fold_df = pd.DataFrame(loo_rh_fold_records)

display(loo_rh_fold_df.round(4))
display(
    pd.DataFrame(
        [
            {"Model": "C1_LOO_RH", **loo_rh_c1_metrics},
            {"Model": "C5_LOO_RH", **loo_rh_c5_metrics},
        ]
    ).round(4)
)

In [ ]:
# ------------------------------------------------------------
# Stress Test B: Leave-one-day-out for C1 and C5
# ------------------------------------------------------------

day_levels = sorted(calibration_summary_df["Day_ID"].unique())

loo_day_predictions_c1 = np.full(len(calibration_summary_df), np.nan, dtype=float)
loo_day_predictions_c5 = np.full(len(calibration_summary_df), np.nan, dtype=float)

loo_day_fold_records = []

for held_out_day in day_levels:
    test_mask = calibration_summary_df["Day_ID"].eq(held_out_day)
    train_mask = ~test_mask

    train_idx = np.flatnonzero(train_mask.to_numpy())
    test_idx = np.flatnonzero(test_mask.to_numpy())

    # C1: training-only global offset
    c1_offset = float(y_correction.iloc[train_idx].mean())
    loo_day_predictions_c1[test_idx] = c1_offset

    # C5: training-only Direction diagnostic
    model_c5_day = LinearRegression()
    model_c5_day.fit(
        X_c5.iloc[train_idx],
        y_correction.iloc[train_idx],
    )
    loo_day_predictions_c5[test_idx] = model_c5_day.predict(
        X_c5.iloc[test_idx]
    )

    residual_c1_fold = (
        calibration_summary_df.iloc[test_idx]["Error_RH_Pre"].to_numpy()
        + loo_day_predictions_c1[test_idx]
    )
    residual_c5_fold = (
        calibration_summary_df.iloc[test_idx]["Error_RH_Pre"].to_numpy()
        + loo_day_predictions_c5[test_idx]
    )

    loo_day_fold_records.append(
        {
            "Held_Out_Day": int(held_out_day),
            "N_Test": len(test_idx),
            "C1_Offset": c1_offset,
            "C1_MBE": float(np.mean(residual_c1_fold)),
            "C1_RMSE": float(np.sqrt(np.mean(residual_c1_fold**2))),
            "C5_DOWN_Correction": float(model_c5_day.intercept_),
            "C5_UP_Increment": float(model_c5_day.coef_[0]),
            "C5_MBE": float(np.mean(residual_c5_fold)),
            "C5_RMSE": float(np.sqrt(np.mean(residual_c5_fold**2))),
        }
    )

assert not np.isnan(loo_day_predictions_c1).any()
assert not np.isnan(loo_day_predictions_c5).any()

loo_day_results_df = calibration_summary_df[
    [
        "Test_ID",
        "RH_Setpoint",
        "Temperature_Setpoint_C",
        "Direction",
        "Day_ID",
        "Run_ID",
        "RH_DUT_Mean",
        "RH_Reference_Mean",
        "Error_RH_Pre",
    ]
].copy()

loo_day_results_df["Predicted_Correction_C1"] = loo_day_predictions_c1
loo_day_results_df["Predicted_Correction_C5"] = loo_day_predictions_c5

loo_day_results_df["Residual_C1"] = (
    loo_day_results_df["Error_RH_Pre"]
    + loo_day_results_df["Predicted_Correction_C1"]
)
loo_day_results_df["Residual_C5"] = (
    loo_day_results_df["Error_RH_Pre"]
    + loo_day_results_df["Predicted_Correction_C5"]
)

loo_day_c1_metrics = calibration_metrics(
    residual=loo_day_results_df["Residual_C1"],
    rh_reference=loo_day_results_df["RH_Reference_Mean"],
    rh_calibrated=(
        loo_day_results_df["RH_DUT_Mean"]
        + loo_day_results_df["Predicted_Correction_C1"]
    ),
)

loo_day_c5_metrics = calibration_metrics(
    residual=loo_day_results_df["Residual_C5"],
    rh_reference=loo_day_results_df["RH_Reference_Mean"],
    rh_calibrated=(
        loo_day_results_df["RH_DUT_Mean"]
        + loo_day_results_df["Predicted_Correction_C5"]
    ),
)

loo_day_fold_df = pd.DataFrame(loo_day_fold_records)

display(loo_day_fold_df.round(4))
display(
    pd.DataFrame(
        [
            {"Model": "C1_LOO_Day", **loo_day_c1_metrics},
            {"Model": "C5_LOO_Day", **loo_day_c5_metrics},
        ]
    ).round(4)
)

## 33. Robustness Summary

The robustness decision should distinguish two ideas:

- **predictive robustness** — whether performance remains similar when an entire RH level or day is withheld;
- **deployability** — whether the required predictors are available under intended field operation.

C1 can be robust yet still limited by hysteresis.

C5 can be highly accurate yet remain non-deployable unless Direction/state is operationally observable.

In [ ]:
# ------------------------------------------------------------
# Compact robustness comparison
# ------------------------------------------------------------

robustness_summary_df = pd.DataFrame(
    [
        {
            "Model": "C1",
            "Primary_GroupCV_RMSE": c1_metrics["RMSE"],
            "LOO_RH_RMSE": loo_rh_c1_metrics["RMSE"],
            "LOO_Day_RMSE": loo_day_c1_metrics["RMSE"],
        },
        {
            "Model": "C5_Diagnostic",
            "Primary_GroupCV_RMSE": c5_metrics["RMSE"],
            "LOO_RH_RMSE": loo_rh_c5_metrics["RMSE"],
            "LOO_Day_RMSE": loo_day_c5_metrics["RMSE"],
        },
    ]
)

display(robustness_summary_df.round(4))

## 34. Robustness Checkpoint

Before freezing a calibration artifact, review:

- C1 leave-one-RH performance versus primary group-CV performance;
- C5 leave-one-RH performance versus primary group-CV performance;
- held-out edge RH levels 20 and 80 %RH;
- C1 leave-one-day MBE and RMSE;
- C5 leave-one-day MBE and RMSE;
- stability of the C5 Direction correction across stress-test fits.

### Freeze decision rule

If C1 remains stable across the stress tests, it can be frozen as the **memoryless deployable calibration candidate**, with an explicit documented hysteresis limitation.

If C5 remains strongly superior across the stress tests, it should be frozen separately as a **diagnostic/state-aware model**, not silently substituted for the deployable model.

The reserved VALIDATION dataset must remain untouched until the calibration model identity and coefficients are frozen.

## 35. Model Freeze Decision

The pre-specified calibration-development evidence now supports a freeze decision.

### Selected memoryless deployable model

```text
C1 — Constant Offset
```

Rationale:

- approximately 60% RMSE reduction relative to the uncalibrated C0 baseline;
- simpler and more stable than C2-C4;
- C2 adds no meaningful RMSE improvement;
- C3 and C4 materially worsen out-of-fold performance;
- C1 remains stable under leave-one-RH and leave-one-day stress tests;
- C1 uses no reference, setpoint, Day, Run, Test_ID, hidden truth, or Direction input at deployment.

### Retained state-aware diagnostic model

```text
C5 — Direction-Augmented Diagnostic
```

Rationale:

- large and robust improvement over C1;
- identifies Direction/hysteresis as the dominant unresolved systematic structure;
- stable Direction increment across primary CV, leave-one-RH, and leave-one-day checks.

C5 is not silently substituted for the field model because Direction is not yet defined as an operationally reliable deployment input.

### Freeze condition

Both models are now refit using all 96 CALIBRATION Test_ID summaries.

The reserved VALIDATION measurements remain untouched.

The canonical deployable freeze record is:

```text
config/calibration_model_v1.json
```

A separate diagnostic record is also exported:

```text
config/calibration_model_direction_diagnostic_v1.json
```

In [ ]:
# ------------------------------------------------------------
# Final refit on ALL 96 CALIBRATION Test_ID summaries
# No VALIDATION measurement values are used.
# ------------------------------------------------------------

assert len(calibration_summary_df) == 96
assert set(calibration_summary_df["Analysis_Role"]) == {"CALIBRATION"}

# C1 final deployable model: one global additive correction.
final_c1_offset = float(calibration_summary_df["Correction_RH"].mean())

# C5 final diagnostic/state-aware model: binary Direction indicator.
final_c5_model = LinearRegression()
final_c5_model.fit(
    X_c5,
    y_correction,
)

final_c5_intercept_down = float(final_c5_model.intercept_)
final_c5_up_increment = float(final_c5_model.coef_[0])
final_c5_correction_up = (
    final_c5_intercept_down + final_c5_up_increment
)

print(f"Final C1 offset: {final_c1_offset:.10f} %RH")
print(
    "Final C5 DOWN correction: "
    f"{final_c5_intercept_down:.10f} %RH"
)
print(
    "Final C5 UP increment: "
    f"{final_c5_up_increment:.10f} %RH"
)
print(
    "Final C5 UP correction: "
    f"{final_c5_correction_up:.10f} %RH"
)

## 36. Canonical Freeze Artifacts

The JSON files below are intentionally human-readable.

The canonical deployment equation is:

```text
RH_Calibrated = RH_DUT + offset
```

where `offset` is the fitted C1 correction from all CALIBRATION Test_ID summaries.

The diagnostic C5 specification records the separate UP/DOWN correction structure, but explicitly labels the model as non-deployable unless Direction can be operationally determined.

In [ ]:
# ------------------------------------------------------------
# Write canonical JSON model freeze records
# ------------------------------------------------------------

config_dir = project_root / "config"
config_dir.mkdir(parents=True, exist_ok=True)

deployable_freeze_record = {
    "model_id": "C1_Constant_Offset",
    "model_version": "1.0",
    "dataset_release": "VSM-LAB-DATA-v1.1",
    "milestone_id": "HV-006",
    "technical_gate_id": "CM-001",
    "role": "memoryless_deployable_calibration",
    "model_form": "RH_Calibrated = RH_DUT + offset",
    "target_definition": (
        "Correction_RH = RH_Reference_Mean - RH_DUT_Mean"
    ),
    "predictors": [],
    "preprocessing": "none",
    "feature_names_after_transformation": [],
    "parameters": {
        "offset_percent_rh": final_c1_offset,
    },
    "regularization": None,
    "calibration_test_id_count": int(len(calibration_summary_df)),
    "modeling_unit": "one summary row per CALIBRATION Test_ID",
    "selection_summary": {
        "selected_over": [
            "C2_RH_Linear",
            "C3_RH_Temperature_Linear",
            "C4_Polynomial_Ridge",
        ],
        "primary_groupcv_rmse_percent_rh": float(c1_metrics["RMSE"]),
        "primary_groupcv_mae_percent_rh": float(c1_metrics["MAE"]),
        "loo_rh_rmse_percent_rh": float(loo_rh_c1_metrics["RMSE"]),
        "loo_day_rmse_percent_rh": float(loo_day_c1_metrics["RMSE"]),
        "documented_limitation": (
            "Strong Direction/hysteresis structure remains in a "
            "memoryless calibration."
        ),
    },
    "software": {
        "python": ".".join(map(str, __import__("sys").version_info[:3])),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "validation_measurements_used": False,
    "validation_statement": (
        "Reserved VALIDATION measurement values were not used for "
        "fitting, tuning, model selection, or model freeze."
    ),
}

diagnostic_freeze_record = {
    "model_id": "C5_Direction_Diagnostic",
    "model_version": "1.0",
    "dataset_release": "VSM-LAB-DATA-v1.1",
    "milestone_id": "HV-006",
    "technical_gate_id": "CM-001",
    "role": "state_aware_diagnostic_not_yet_deployable",
    "model_form": (
        "Correction_RH = intercept_down + "
        "up_increment * I(Direction == UP)"
    ),
    "target_definition": (
        "Correction_RH = RH_Reference_Mean - RH_DUT_Mean"
    ),
    "predictors": ["Direction_UP"],
    "preprocessing": {
        "Direction_UP": "1 for UP, 0 for DOWN"
    },
    "feature_names_after_transformation": ["Direction_UP"],
    "parameters": {
        "intercept_down_correction_percent_rh": final_c5_intercept_down,
        "up_increment_percent_rh": final_c5_up_increment,
        "up_correction_percent_rh": final_c5_correction_up,
    },
    "regularization": None,
    "calibration_test_id_count": int(len(calibration_summary_df)),
    "modeling_unit": "one summary row per CALIBRATION Test_ID",
    "selection_summary": {
        "primary_groupcv_rmse_percent_rh": float(c5_metrics["RMSE"]),
        "primary_groupcv_mae_percent_rh": float(c5_metrics["MAE"]),
        "loo_rh_rmse_percent_rh": float(loo_rh_c5_metrics["RMSE"]),
        "loo_day_rmse_percent_rh": float(loo_day_c5_metrics["RMSE"]),
        "interpretation": (
            "Direction/hysteresis contains major calibration information."
        ),
        "deployment_condition": (
            "Do not use as the field calibration model unless Direction "
            "can be operationally and reliably determined."
        ),
    },
    "software": {
        "python": ".".join(map(str, __import__("sys").version_info[:3])),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "validation_measurements_used": False,
    "validation_statement": (
        "Reserved VALIDATION measurement values were not used for "
        "fitting, tuning, model selection, or model freeze."
    ),
}

deployable_path = config_dir / "calibration_model_v1.json"
diagnostic_path = (
    config_dir / "calibration_model_direction_diagnostic_v1.json"
)

deployable_path.write_text(
    json.dumps(deployable_freeze_record, indent=2) + "\n",
    encoding="utf-8",
)
diagnostic_path.write_text(
    json.dumps(diagnostic_freeze_record, indent=2) + "\n",
    encoding="utf-8",
)

print(f"Wrote: {deployable_path.relative_to(project_root)}")
print(f"Wrote: {diagnostic_path.relative_to(project_root)}")

## 37. Curated HV-006 Evidence Exports

The model-development evidence is exported in compact CSV form for the final report and milestone handoff.

In [ ]:
# ------------------------------------------------------------
# Curated model-development evidence exports
# ------------------------------------------------------------

tables_dir = project_root / "reports" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

model_comparison_export = c0_c5_comparison_df.copy()

selected_model_coefficients_df = pd.DataFrame(
    [
        {
            "Model": "C1_Constant_Offset",
            "Parameter": "offset_percent_rh",
            "Value": final_c1_offset,
            "Role": "Deployable",
        },
        {
            "Model": "C5_Direction_Diagnostic",
            "Parameter": "intercept_down_correction_percent_rh",
            "Value": final_c5_intercept_down,
            "Role": "Diagnostic",
        },
        {
            "Model": "C5_Direction_Diagnostic",
            "Parameter": "up_increment_percent_rh",
            "Value": final_c5_up_increment,
            "Role": "Diagnostic",
        },
        {
            "Model": "C5_Direction_Diagnostic",
            "Parameter": "up_correction_percent_rh",
            "Value": final_c5_correction_up,
            "Role": "Diagnostic",
        },
    ]
)

selected_model_residual_summary_df = pd.DataFrame(
    [
        {
            "Model": "C1_Constant_Offset",
            "Assessment": "Primary_GroupCV",
            **c1_metrics,
        },
        {
            "Model": "C1_Constant_Offset",
            "Assessment": "LOO_RH",
            **loo_rh_c1_metrics,
        },
        {
            "Model": "C1_Constant_Offset",
            "Assessment": "LOO_Day",
            **loo_day_c1_metrics,
        },
        {
            "Model": "C5_Direction_Diagnostic",
            "Assessment": "Primary_GroupCV",
            **c5_metrics,
        },
        {
            "Model": "C5_Direction_Diagnostic",
            "Assessment": "LOO_RH",
            **loo_rh_c5_metrics,
        },
        {
            "Model": "C5_Direction_Diagnostic",
            "Assessment": "LOO_Day",
            **loo_day_c5_metrics,
        },
    ]
)

model_comparison_export.to_csv(
    tables_dir / "hv006_model_comparison.csv",
    index=False,
)
fold_metrics_export = pd.concat(
    [
        loo_rh_fold_df.assign(Stress_Test="Leave_One_RH"),
        loo_day_fold_df.assign(Stress_Test="Leave_One_Day"),
    ],
    ignore_index=True,
    sort=False,
)
fold_metrics_export.to_csv(
    tables_dir / "hv006_cv_fold_metrics.csv",
    index=False,
)
selected_model_residual_summary_df.to_csv(
    tables_dir / "hv006_selected_model_residual_summary.csv",
    index=False,
)
selected_model_coefficients_df.to_csv(
    tables_dir / "hv006_selected_model_coefficients.csv",
    index=False,
)

print("Curated HV-006 tables exported.")

## 38. Model Freeze Checkpoint

The model-freeze checkpoint passes only if all of the following are true:

- C1 is refit using all 96 CALIBRATION Test_ID summaries;
- C5 diagnostic is refit using all 96 CALIBRATION Test_ID summaries;
- `config/calibration_model_v1.json` is created;
- `config/calibration_model_direction_diagnostic_v1.json` is created;
- both JSON records state that VALIDATION measurements were not used;
- curated comparison, stress-test, residual-summary, and coefficient CSVs are written;
- the notebook reproduces from `Restart Kernel -> Run All` without error.

At this point **do not evaluate the reserved VALIDATION measurements yet**.

First inspect the generated JSON freeze records and exact fitted parameters. Once they are accepted and committed, CM-001 can be closed and the independent-validation milestone can begin.